<a href="https://colab.research.google.com/github/drhigh4444-code/Agentic-AI/blob/main/LLM_Planning_Systems_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM Internals & Planning Systems — Live Lab
### MSAGI · Applied Agentic AI · Simplilearn

**What we build:** a single agent that takes a goal in plain English, **plans** the steps,
**calls a tool** when it needs one, observes the result, and **loops until done** — the
ReAct pattern (Reason + Act) from the slides.

We write the loop **in plain Python** so you can see the mechanism directly, rather than
hiding it inside a framework. The only dependency is the official model SDK.

> Run each cell top to bottom (Shift+Enter). The printed **Thought / Action / Observation**
> trace IS the lesson — you are watching the model plan.


## Step 1 — Install the SDK
One small dependency. ~10 seconds.

In [1]:
!pip install -q openai
print("Ready.")

Ready.


## Step 2 — Set your API key
Paste the key into the hidden prompt when it appears. It is **not** stored in the notebook,
and it will **not** appear on your shared screen.

> Teaching note: any function-calling model works. We keep this vendor-neutral — the
> *planning behavior* is the point, not the model brand.

In [6]:
import os, getpass
os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key (hidden): ")
from openai import OpenAI
client = OpenAI()
MODEL = "gpt-4o-mini"   # any tool-calling model is fine
print("Client ready.")

Enter API key (hidden): ··········
Client ready.


## Step 3 — Define two simple tools
A tool is just a Python function the agent is allowed to call. We give it:
1. a **calculator** (LLMs are unreliable at arithmetic — so we offload it), and
2. a **word_counter** (a stand-in for any real API: a CRM lookup, an inventory check, etc.).

The **description** in each schema is not decoration — the model reads it to decide
*when* to use the tool. In function calling, the description is the interface.

In [3]:
def calculator(expression: str) -> str:
    """Evaluate a math expression like '128 * 0.15'."""
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"error: {e}"

def word_counter(text: str) -> str:
    """Return the number of words in a piece of text."""
    return str(len(text.split()))

TOOLS = {"calculator": calculator, "word_counter": word_counter}

# The schemas the model sees. NOTE how much the description matters.
TOOL_SCHEMAS = [
    {"type": "function", "function": {
        "name": "calculator",
        "description": "Evaluate an arithmetic expression. Use this for ANY math - the model is unreliable at arithmetic on its own.",
        "parameters": {"type": "object",
            "properties": {"expression": {"type": "string", "description": "e.g. '9 * 12'"}},
            "required": ["expression"]}}},
    {"type": "function", "function": {
        "name": "word_counter",
        "description": "Count the number of words in a piece of text.",
        "parameters": {"type": "object",
            "properties": {"text": {"type": "string"}},
            "required": ["text"]}}},
]
print("Tools registered:", list(TOOLS))

Tools registered: ['calculator', 'word_counter']


## Step 4 — The ReAct loop (write it ourselves)
This is the whole engine, in ~15 lines. Watch what it does:
**ask the model -> if it wants a tool, run it and feed the result back -> repeat -> until it answers.**
That loop is *agentic reasoning*. Everything else in the course builds on it.

In [4]:
import json

def run_agent(task, max_steps=6, verbose=True):
    messages = [
        {"role": "system", "content": "You are a planning agent. Break the goal into steps and use tools when helpful."},
        {"role": "user", "content": task},
    ]
    if verbose: print(f"GOAL: {task}\n" + "-"*60)
    for step in range(max_steps):
        resp = client.chat.completions.create(
            model=MODEL, messages=messages, tools=TOOL_SCHEMAS, temperature=0)
        msg = resp.choices[0].message
        if msg.tool_calls:
            messages.append(msg.model_dump())          # record the model's plan
            for tc in msg.tool_calls:
                name = tc.function.name
                args = json.loads(tc.function.arguments)
                if verbose: print(f"Thought:     I should use `{name}`.")
                result = TOOLS[name](**args)            # ACT: run the tool
                if verbose: print(f"Action:      {name}({args})")
                if verbose: print(f"Observation: {result}\n")
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": str(result)})
        else:
            if verbose: print(f"Final Answer: {msg.content}")
            return msg.content
    return "(stopped: hit max steps)"

print("Agent defined.")

Agent defined.


## Step 5 — Run it: a goal that REQUIRES a plan
This can't be done in one shot. The agent must (1) count the words, (2) multiply by 12,
(3) report the answer. **Watch the trace** — each Thought is the model planning, each
Action is a tool call, each Observation feeds the next step.

In [7]:
task = ("Count the number of words in this sentence: "
        "'Planning systems turn a prompt into a sequence of actions'. "
        "Then multiply that word count by 12 and give me the final number.")

answer = run_agent(task)
print("\n=== RETURNED ===")
print(answer)

GOAL: Count the number of words in this sentence: 'Planning systems turn a prompt into a sequence of actions'. Then multiply that word count by 12 and give me the final number.
------------------------------------------------------------
Thought:     I should use `word_counter`.
Action:      word_counter({'text': 'Planning systems turn a prompt into a sequence of actions'})
Observation: 10

Thought:     I should use `calculator`.
Action:      calculator({'expression': '10 * 12'})
Observation: 120

Final Answer: The final number, after counting the words in the sentence and multiplying by 12, is 120.

=== RETURNED ===
The final number, after counting the words in the sentence and multiplying by 12, is 120.


## Step 6 — Guided practice (your turn)
Change the goal and re-run. Try one that needs **both** tools, or one that needs **none**
(watch the agent correctly decide *not* to call a tool — restraint is also planning).

**Discussion:** where would this break? An ambiguous goal? A tool that errors? That
question is the bridge to *guardrails* and to *multi-agent* — next session.

In [8]:
your_task = "TODO: write your own multi-step goal here"

# Uncomment to run:
# print(run_agent(your_task))

## Key takeaways
- A **plan** is the model decomposing a goal into ordered steps.
- **Function calling** lets the model act through typed tools — the *description* is the contract.
- The **ReAct loop** (Thought -> Action -> Observation -> repeat) is the simplest durable
  single-agent pattern. You just built it in ~15 lines.
- Product lens: the trace you watched is also your **observability surface** — what you log,
  evaluate, and debug in production (LangSmith & friends, later modules).

**Next session:** one planner is good — what happens when you need *several* specialized
agents working together? -> **Multi-Agent Ecosystems (I).**
